# Data Preprocessing and Cleaning

This notebook builds one row per order for two late-delivery tasks:

- **Classification:** predict whether a delivered order arrives after its promised calendar date (`is_late`).
- **Regression:** predict elapsed time from purchase to delivery in days (`delivery_days`).

The prediction point is the time of purchase. Reviews, final order status, carrier and actual-delivery events are therefore excluded from model features.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').is_dir():
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

CUTOFF_DATE = pd.Timestamp('2018-08-31 23:59:59')
TRAIN_END = pd.Timestamp('2018-06-30 23:59:59')
VALID_END = pd.Timestamp('2018-07-31 23:59:59')

## 1. Load the raw tables

Review data are deliberately not loaded because reviews occur after delivery and would leak the outcome. Geolocation is aggregated to one median coordinate per zip-code prefix before joining, preventing the raw table's duplicate zip prefixes from multiplying order rows.

In [2]:
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
orders = pd.read_csv(RAW / 'olist_orders_dataset.csv', parse_dates=date_columns)
items = pd.read_csv(RAW / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
products = pd.read_csv(RAW / 'olist_products_dataset.csv')
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')
sellers = pd.read_csv(RAW / 'olist_sellers_dataset.csv')
payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
translations = pd.read_csv(RAW / 'product_category_name_translation.csv')
geolocation = pd.read_csv(RAW / 'olist_geolocation_dataset.csv')

pd.DataFrame({
    'rows': [len(x) for x in [orders, items, products, customers, sellers, payments, geolocation]],
    'unique_orders': [orders.order_id.nunique(), items.order_id.nunique(), np.nan, np.nan, np.nan, payments.order_id.nunique(), np.nan],
}, index=['orders', 'items', 'products', 'customers', 'sellers', 'payments', 'geolocation'])

,rows,unique_orders
orders,99441,99441.0
items,112650,98666.0
products,32951,NaN
customers,99441,NaN
sellers,3095,NaN
payments,103886,99440.0
geolocation,1000163,NaN


## 2. Confirm the modelling grain

Actual and estimated delivery timestamps exist only in the orders table. Item rows may represent separate products or sellers, but there is no item/package-level actual delivery timestamp. The modelling grain must therefore be one row per `order_id`.

In [3]:
item_grain = items.groupby('order_id').agg(
    item_rows=('order_item_id', 'size'),
    products=('product_id', 'nunique'),
    sellers=('seller_id', 'nunique'),
    shipping_deadlines=('shipping_limit_date', 'nunique'),
)
grain_check = pd.Series({
    'orders_with_items': len(item_grain),
    'multi_item_orders': int(item_grain.item_rows.gt(1).sum()),
    'multi_product_orders': int(item_grain.products.gt(1).sum()),
    'multi_seller_orders': int(item_grain.sellers.gt(1).sum()),
    'orders_with_multiple_shipping_deadlines': int(item_grain.shipping_deadlines.gt(1).sum()),
})
display(grain_check.to_frame('orders'))
assert not orders.order_id.duplicated().any()
assert not items.duplicated(['order_id', 'order_item_id']).any()

,orders
orders_with_items,98666
multi_item_orders,9803
multi_product_orders,3236
multi_seller_orders,1278
orders_with_multiple_shipping_deadlines,336


## 3. Aggregate purchase-time information to order level

Product measurements are summed across item rows to approximate shipment size. Geolocation rows outside Brazil's coordinate bounds are removed, and duplicate zip-code prefixes are reduced to median coordinates. Seller-to-customer great-circle distances are later aggregated across each order's unique sellers. Orders containing an item with missing measurements are flagged and removed later because an order total would otherwise be understated. Missing categories are retained as `Unknown`.

In [4]:
product_columns = [
    'product_id', 'product_category_name', 'product_weight_g',
    'product_length_cm', 'product_height_cm', 'product_width_cm',
]
product_lookup = products[product_columns].merge(
    translations, on='product_category_name', how='left', validate='many_to_one'
)
geo_clean = geolocation.loc[
    geolocation.geolocation_lat.between(-34, 6)
    & geolocation.geolocation_lng.between(-74, -34)
]
geo_by_zip = geo_clean.groupby('geolocation_zip_code_prefix', as_index=False).agg(
    latitude=('geolocation_lat', 'median'),
    longitude=('geolocation_lng', 'median'),
)
assert geo_by_zip.geolocation_zip_code_prefix.is_unique
seller_lookup = (
    sellers.merge(geo_by_zip, left_on='seller_zip_code_prefix',
                  right_on='geolocation_zip_code_prefix', how='left', validate='many_to_one')
    .rename(columns={'latitude': 'seller_zip_lat', 'longitude': 'seller_zip_lng'})
)
item_data = (
    items.merge(product_lookup, on='product_id', how='left', validate='many_to_one')
    .merge(seller_lookup[['seller_id', 'seller_state', 'seller_zip_lat', 'seller_zip_lng']],
           on='seller_id', how='left', validate='many_to_one')
)
item_data['product_category'] = item_data.product_category_name_english.fillna('Unknown')
item_data['product_volume_cm3'] = (
    item_data.product_length_cm * item_data.product_height_cm * item_data.product_width_cm
)
item_data['missing_product_measurement'] = item_data[
    ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
].isna().any(axis=1)

order_items = item_data.sort_values(['order_id', 'order_item_id']).groupby('order_id', as_index=False).agg(
    order_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    item_count=('order_item_id', 'size'),
    unique_product_count=('product_id', 'nunique'),
    seller_count=('seller_id', 'nunique'),
    seller_state_count=('seller_state', 'nunique'),
    product_category_count=('product_category', 'nunique'),
    product_category=('product_category', 'first'),
    primary_seller_state=('seller_state', 'first'),
    total_weight_g=('product_weight_g', 'sum'),
    total_volume_cm3=('product_volume_cm3', 'sum'),
    missing_product_measurement=('missing_product_measurement', 'sum'),
)
order_sellers = item_data.drop_duplicates(['order_id', 'seller_id'])[
    ['order_id', 'seller_id', 'seller_zip_lat', 'seller_zip_lng']
]

payment_summary = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type_count=('payment_type', 'nunique'),
    payment_type=('payment_type', 'first'),
)

## 4. Define the eligible delivery cohort and targets

Rules:

1. Keep successfully delivered orders with observed actual and estimated delivery dates.
2. Cut purchase dates at 31 August 2018. September and October contain no delivered cohorts.
3. Define late delivery using calendar dates, not timestamps. Estimated dates are stored at midnight, so timestamp comparison would incorrectly classify delivery during the promised day as late.
4. Keep unresolved, cancelled and unavailable orders out of the late/on-time target; they are not negative examples.

In [5]:
status_audit = orders.groupby('order_status').agg(
    orders=('order_id', 'size'),
    missing_actual_delivery=('order_delivered_customer_date', lambda x: x.isna().sum()),
    last_purchase=('order_purchase_timestamp', 'max'),
).sort_values('orders', ascending=False)
display(status_audit)

eligible_orders = orders.loc[
    orders.order_status.eq('delivered')
    & orders.order_delivered_customer_date.notna()
    & orders.order_estimated_delivery_date.notna()
    & orders.order_purchase_timestamp.le(CUTOFF_DATE)
].copy()

eligible_orders['delivery_days'] = (
    eligible_orders.order_delivered_customer_date - eligible_orders.order_purchase_timestamp
).dt.total_seconds().div(86400)
eligible_orders['is_late'] = (
    eligible_orders.order_delivered_customer_date.dt.normalize()
    > eligible_orders.order_estimated_delivery_date.dt.normalize()
).astype('int8')
eligible_orders['promised_delivery_days'] = (
    eligible_orders.order_estimated_delivery_date.dt.normalize()
    - eligible_orders.order_purchase_timestamp.dt.normalize()
).dt.days

timestamp_late = (
    eligible_orders.order_delivered_customer_date > eligible_orders.order_estimated_delivery_date
).sum()
pd.Series({
    'eligible_delivered_orders': len(eligible_orders),
    'late_by_calendar_date': int(eligible_orders.is_late.sum()),
    'late_rate': eligible_orders.is_late.mean(),
    'late_by_timestamp': int(timestamp_late),
    'same_promised_day_false_lates_avoided': int(timestamp_late - eligible_orders.is_late.sum()),
})

,orders,missing_actual_delivery,last_purchase
order_status,,,
delivered,96478,8,2018-08-29 15:00:37
shipped,1107,1107,2018-09-03 09:06:57
canceled,625,619,2018-10-17 17:30:18
unavailable,609,609,2018-08-21 12:21:00
invoiced,314,314,2018-08-14 18:45:08
processing,301,301,2018-07-23 18:03:03
created,5,5,2018-02-09 17:21:04
approved,2,2,2017-04-25 01:25:34


eligible_delivered_orders                96470.000000
late_by_calendar_date                     6534.000000
late_rate                                    0.067731
late_by_timestamp                         7826.000000
same_promised_day_false_lates_avoided     1292.000000
dtype: float64

## 5. Join and clean the order-level modelling table

No global imputation, encoding, scaling or outlier clipping is performed here. Those transformations must be fitted on the training set inside the later modelling pipelines.

In [6]:
customer_lookup = (
    customers.merge(geo_by_zip, left_on='customer_zip_code_prefix',
                    right_on='geolocation_zip_code_prefix', how='left', validate='many_to_one')
    .rename(columns={'latitude': 'customer_zip_lat', 'longitude': 'customer_zip_lng'})
)
customer_columns = [
    'customer_id', 'customer_state', 'customer_zip_code_prefix',
    'customer_zip_lat', 'customer_zip_lng',
]
model_data = (
    eligible_orders
    .merge(customer_lookup[customer_columns], on='customer_id', how='left', validate='many_to_one')
    .merge(order_items, on='order_id', how='inner', validate='one_to_one')
    .merge(payment_summary, on='order_id', how='left', validate='one_to_one')
)
routes = order_sellers.merge(
    model_data[['order_id', 'customer_zip_lat', 'customer_zip_lng']],
    on='order_id', how='inner', validate='many_to_one',
)
lat1, lon1, lat2, lon2 = map(np.radians, [
    routes.customer_zip_lat, routes.customer_zip_lng,
    routes.seller_zip_lat, routes.seller_zip_lng,
])
routes['estimated_seller_customer_distance_km'] = 6371.0088 * 2 * np.arcsin(np.sqrt(
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
))
order_geo = routes.groupby('order_id', as_index=False).agg(
    mean_seller_zip_lat=('seller_zip_lat', 'mean'),
    mean_seller_zip_lng=('seller_zip_lng', 'mean'),
    estimated_min_distance_km=('estimated_seller_customer_distance_km', 'min'),
    estimated_mean_distance_km=('estimated_seller_customer_distance_km', 'mean'),
    estimated_max_distance_km=('estimated_seller_customer_distance_km', 'max'),
    located_seller_count=('estimated_seller_customer_distance_km', 'count'),
)
model_data = model_data.merge(order_geo, on='order_id', how='left', validate='one_to_one')
model_data['geolocation_missing'] = (
    model_data.customer_zip_lat.isna() | model_data.located_seller_count.lt(model_data.seller_count)
).astype('int8')
model_data['same_state'] = (model_data.customer_state == model_data.primary_seller_state).astype('int8')

cleaning_audit = pd.Series({
    'eligible_before_product_cleaning': len(model_data),
    'orders_with_missing_product_measurements': int(model_data.missing_product_measurement.gt(0).sum()),
    'orders_with_missing_payment_summary': int(model_data.payment_value.isna().sum()),
    'orders_with_incomplete_geolocation': int(model_data.geolocation_missing.sum()),
    'non_positive_delivery_duration': int(model_data.delivery_days.le(0).sum()),
})
display(cleaning_audit.to_frame('orders'))

model_data = model_data.loc[
    model_data.missing_product_measurement.eq(0) & model_data.delivery_days.gt(0)
].copy()
model_data['payment_missing'] = model_data.payment_value.isna().astype('int8')
model_data['payment_type'] = model_data.payment_type.fillna('Unknown')
model_data['product_category'] = model_data.product_category.fillna('Unknown')
model_data['purchase_month'] = model_data.order_purchase_timestamp.dt.to_period('M').astype(str)

,orders
eligible_before_product_cleaning,96470
orders_with_missing_product_measurements,16
orders_with_missing_payment_summary,1
orders_with_incomplete_geolocation,481
non_positive_delivery_duration,0


## 6. Outlier diagnostics

Extreme delivery times and shipment values are retained. Removing or capping them before the split would use information from the validation and test sets. Robust transformations, if needed, belong in the downstream training pipeline.

In [7]:
diagnostic_columns = [
    'delivery_days', 'order_value', 'freight_value',
    'total_weight_g', 'total_volume_cm3', 'item_count',
]
model_data[diagnostic_columns].describe(percentiles=[0.01, 0.5, 0.95, 0.99, 0.999]).T

,count,mean,std,min,1%,50%,95%,99%,99.9%,max
delivery_days,96454.0,12.558280,9.546432,0.533414,1.825014,10.218119,29.273926,46.051191,83.145268,2.096286e+02
order_value,96454.0,137.036846,208.986117,0.850000,11.990000,86.615000,399.000000,990.000000,2258.000000,1.344000e+04
freight_value,96454.0,22.786724,21.561466,0.000000,7.390000,17.170000,54.787000,104.249400,232.881450,1.794960e+03
total_weight_g,96454.0,2386.868310,4770.109876,0.000000,100.000000,750.000000,10500.000000,22350.000000,47932.050000,1.844000e+05
total_volume_cm3,96454.0,17331.936799,30143.870593,168.000000,352.000000,7254.000000,64000.000000,135690.820000,294000.000000,1.476000e+06
item_count,96454.0,1.142213,0.538852,1.000000,1.000000,1.000000,2.000000,3.000000,6.000000,2.100000e+01


## 7. Create leakage-safe classification and regression tables

`order_id` is retained only for auditing and must not be passed to a model. `order_purchase_timestamp` and `purchase_month` support the time split; later feature engineering may derive purchase-time calendar features from them.

### 7.1 Data dictionary for the exported modelling tables

All predictors below are known at purchase time. Coordinates are **median coordinates of a zip-code-prefix area**, not exact customer or seller addresses. Distances are Haversine straight-line estimates between those representative coordinates.

| Column | English meaning | 中文说明 |
|---|---|---|
| `order_id` | Order identifier retained for auditing only; exclude it from model inputs. | 订单唯一标识，仅用于审计和核对，不应作为模型特征。 |
| `order_purchase_timestamp` | Purchase date and time; used for chronological splitting and purchase-time features. | 下单日期和时间，用于按时间划分数据并生成购买时间特征。 |
| `purchase_month` | Purchase year-month in `YYYY-MM` format. | 下单年月，格式为 `YYYY-MM`。 |
| `customer_state` | Customer's registered Brazilian state code. | 客户注册地所在的巴西州代码。 |
| `customer_zip_code_prefix` | First five digits of the customer's postal code. | 客户邮政编码的前五位。 |
| `customer_zip_lat`, `customer_zip_lng` | Median latitude and longitude of the customer's zip-code-prefix area. | 客户邮编前缀区域的中位纬度和经度，并非客户精确地址。 |
| `order_value` | Sum of item prices in the order, excluding freight. | 订单中所有商品价格之和，不包含运费。 |
| `freight_value` | Sum of item-level freight charges in the order. | 订单中各商品运费之和。 |
| `item_count` | Number of order-item rows. | 订单包含的商品明细行数。 |
| `unique_product_count` | Number of distinct products in the order. | 订单中不同商品的数量。 |
| `seller_count` | Number of distinct sellers in the order. | 订单中不同卖家的数量。 |
| `seller_state_count` | Number of distinct seller states in the order. | 订单卖家涉及的不同州数量。 |
| `product_category_count` | Number of distinct product categories in the order. | 订单中不同商品类别的数量。 |
| `product_category` | Category of the first order item; `Unknown` when no translation is available. | 订单第一个商品的类别；无英文翻译时记为 `Unknown`。 |
| `primary_seller_state` | State of the seller for the first order item. | 订单第一个商品所属卖家的州。 |
| `same_state` | 1 when the customer and primary seller are in the same state, otherwise 0. | 客户与主卖家同州时为 1，否则为 0。 |
| `mean_seller_zip_lat`, `mean_seller_zip_lng` | Mean representative coordinates across the order's distinct sellers. | 订单中所有不同卖家邮编代表坐标的平均纬度和经度。 |
| `estimated_min_distance_km` | Shortest estimated customer-to-seller distance among distinct sellers. | 客户与订单卖家之间最短的估算直线距离（公里）。 |
| `estimated_mean_distance_km` | Mean estimated customer-to-seller distance among distinct sellers. | 客户与订单中各不同卖家之间的平均估算直线距离。 |
| `estimated_max_distance_km` | Longest estimated customer-to-seller distance among distinct sellers. | 客户与订单卖家之间最长的估算直线距离。 |
| `geolocation_missing` | 1 when the customer or at least one seller lacks a valid representative coordinate. | 客户或至少一个卖家缺少有效邮编代表坐标时为 1。 |
| `total_weight_g` | Sum of product weight across all item rows. | 订单所有商品重量之和，单位为克。 |
| `total_volume_cm3` | Sum of product length × height × width across all item rows. | 订单所有商品体积之和，体积由长 × 高 × 宽计算，单位为立方厘米。 |
| `payment_value` | Total recorded payment value for the order. | 订单记录的付款总金额。 |
| `payment_installments` | Maximum number of instalments across the order's payment records. | 订单各付款记录中的最大分期期数。 |
| `payment_type_count` | Number of distinct payment types used. | 订单使用的不同付款方式数量。 |
| `payment_type` | First recorded payment type; `Unknown` when the payment summary is missing. | 第一条付款记录的付款方式；缺少付款摘要时记为 `Unknown`。 |
| `payment_missing` | 1 when no payment summary matched the order. | 没有匹配到付款摘要时为 1。 |
| `promised_delivery_days` | Calendar days from purchase date to the promised delivery date. | 从下单日期到承诺送达日期的日历天数。 |
| `split` | Chronological partition: `train`, `validation`, or `test`. | 按时间划分的数据集：训练集、验证集或测试集。 |
| `is_late` | Classification target: 1 if delivered after the promised calendar date, otherwise 0. Present only in the classification file. | 分类目标：实际送达日期晚于承诺日期时为 1，否则为 0；仅出现在分类数据中。 |
| `delivery_days` | Regression target: elapsed days from purchase timestamp to actual delivery timestamp. Present only in the regression file. | 回归目标：从下单时间到实际送达时间的天数；仅出现在回归数据中。 |

In [8]:
feature_columns = [
    'order_id', 'order_purchase_timestamp', 'purchase_month',
    'customer_state', 'customer_zip_code_prefix', 'customer_zip_lat', 'customer_zip_lng',
    'order_value', 'freight_value', 'item_count', 'unique_product_count',
    'seller_count', 'seller_state_count', 'product_category_count',
    'product_category', 'primary_seller_state', 'same_state',
    'mean_seller_zip_lat', 'mean_seller_zip_lng',
    'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km',
    'geolocation_missing',
    'total_weight_g', 'total_volume_cm3',
    'payment_value', 'payment_installments', 'payment_type_count',
    'payment_type', 'payment_missing', 'promised_delivery_days',
]

model_data['split'] = np.select(
    [
        model_data.order_purchase_timestamp.le(TRAIN_END),
        model_data.order_purchase_timestamp.le(VALID_END),
    ],
    ['train', 'validation'],
    default='test',
)

classification_data = model_data[feature_columns + ['split', 'is_late']].copy()
regression_data = model_data[feature_columns + ['split', 'delivery_days']].copy()

display(classification_data.groupby('split').agg(
    orders=('order_id', 'size'), late_orders=('is_late', 'sum'), late_rate=('is_late', 'mean')
))
display(regression_data.groupby('split').delivery_days.agg(['count', 'mean', 'median', 'max']))

,orders,late_orders,late_rate
split,,,
test,6351,393,0.061880
train,83947,5933,0.070676
validation,6156,208,0.033788


,count,mean,median,max
split,,,,
test,6351,7.732780,7.013704,50.841620
train,83947,13.187322,10.958079,209.628611
validation,6156,8.958636,7.501725,77.731146


## 8. Validation and export

The assertions below fail if order duplication or prohibited leakage fields are introduced. The compressed outputs are reproducible intermediate datasets for the two modelling teams.

In [9]:
prohibited = {
    'customer_id', 'order_status', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'shipping_limit_date',
    'delivery_status', 'delay_days', 'review_score',
}
assert classification_data.order_id.is_unique
assert regression_data.order_id.is_unique
assert not prohibited.intersection(classification_data.columns)
assert not prohibited.intersection(regression_data.columns)
assert classification_data.is_late.isin([0, 1]).all()
assert regression_data.delivery_days.gt(0).all()
assert geo_by_zip.latitude.between(-34, 6).all()
assert geo_by_zip.longitude.between(-74, -34).all()
assert classification_data.estimated_mean_distance_km.dropna().between(0, 6000).all()
assert set(classification_data.split.unique()) == {'train', 'validation', 'test'}

classification_path = PROCESSED / 'late_delivery_classification.csv.gz'
regression_path = PROCESSED / 'delivery_time_regression.csv.gz'
classification_data.to_csv(classification_path, index=False, compression='gzip')
regression_data.to_csv(regression_path, index=False, compression='gzip')

print(f'Classification: {classification_data.shape} -> {classification_path.relative_to(ROOT)}')
print(f'Regression:     {regression_data.shape} -> {regression_path.relative_to(ROOT)}')

Classification: (96454, 33) -> data/processed/late_delivery_classification.csv.gz
Regression:     (96454, 33) -> data/processed/delivery_time_regression.csv.gz
